# Bearing Predictive Maintenance — ML Project

**Objective:** classify bearing condition from vibration-derived features and compare multiple machine-learning models.

> **Data note:** The CSV currently provided in this conversation is a **synthetic convenience dataset** modeled on the structure of bearing-vibration features. It is **not the original CWRU measurements**. Replace `DATA_PATH` with a real CWRU-derived CSV before presenting the project as real experimental-data research.

This notebook includes:
- Data loading and validation
- Exploratory data analysis
- Distribution, box, scatter, correlation, and class-balance visualizations
- Train/test split
- Feature scaling
- Logistic Regression, KNN, SVM, Random Forest, Gradient Boosting, and XGBoost (if installed)
- Cross-validation
- Confusion matrices and classification reports
- Feature importance
- Model comparison
- ROC curves where applicable
- Saving the best model


In [ ]:
# 1. Install optional packages if needed
# Run this cell once in a fresh environment.
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib xgboost


In [ ]:
# 2. Imports and plotting setup
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


In [ ]:
# 3. Load dataset
# If your CSV is in the same folder as this notebook, this works directly.
DATA_PATH = Path("bearing_predictive_maintenance_ml_ready.csv")

if not DATA_PATH.exists():
    # Convenient fallback for the file generated in this ChatGPT session
    fallback = Path("/mnt/data/bearing_predictive_maintenance_ml_ready.csv")
    if fallback.exists():
        DATA_PATH = fallback

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# 4. Basic data-quality checks
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe(include="all").T)


In [ ]:
# 5. Clean basic issues
df = df.drop_duplicates().copy()

# Numeric columns: replace infinities and fill rare missing values with median
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Ensure target is categorical text
df["fault_type"] = df["fault_type"].astype(str).str.strip()

print("Cleaned shape:", df.shape)
print("Classes:", df["fault_type"].unique())


## Exploratory Data Analysis

In [ ]:
# 6. Class distribution
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x="fault_type", order=df["fault_type"].value_counts().index)
plt.title("Bearing Fault Class Distribution")
plt.xlabel("Bearing condition")
plt.ylabel("Number of samples")
plt.xticks(rotation=20)
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()


In [ ]:
# 7. Histograms of important vibration features
features_to_plot = ["rms", "std", "peak", "kurtosis", "crest_factor", "shape_factor"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.ravel(), features_to_plot):
    sns.histplot(data=df, x=feature, hue="fault_type", kde=True,
                 element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title(f"Distribution: {feature}")
plt.tight_layout()
plt.show()


In [ ]:
# 8. Boxplots: compare vibration features by fault type
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.ravel(), features_to_plot):
    sns.boxplot(data=df, x="fault_type", y=feature, ax=ax)
    ax.set_title(f"{feature} by bearing condition")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# 9. Scatter plot: RMS vs kurtosis
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x="rms", y="kurtosis",
    hue="fault_type", style="fault_type", alpha=0.65
)
plt.title("RMS vs Kurtosis")
plt.xlabel("RMS vibration")
plt.ylabel("Kurtosis")
plt.tight_layout()
plt.show()


In [ ]:
# 10. Scatter plot: RPM vs RMS
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x="rpm", y="rms",
    hue="fault_type", alpha=0.65
)
plt.title("RPM vs RMS Vibration")
plt.xlabel("RPM")
plt.ylabel("RMS vibration")
plt.tight_layout()
plt.show()


In [ ]:
# 11. Correlation heatmap
corr_cols = [c for c in df.select_dtypes(include=np.number).columns if c != "sample_id"]
plt.figure(figsize=(13, 9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# 12. Grouped means — useful for engineering interpretation
group_means = df.groupby("fault_type")[features_to_plot].mean().round(4)
display(group_means)

group_means.plot(kind="bar", figsize=(13, 6))
plt.title("Mean Vibration Features by Bearing Condition")
plt.xlabel("Bearing condition")
plt.ylabel("Mean feature value")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## ML Preparation

We predict `fault_type`.

`sample_id` is an identifier and should not be used as a predictive feature. `fault_type` is the target. The remaining numeric columns are candidate predictors.

In [ ]:
# 13. Prepare X and y
target = "fault_type"
drop_cols = ["sample_id", target]

X = df.drop(columns=drop_cols)
y = df[target]

# Keep numeric predictors for this first ML pipeline
X = X.select_dtypes(include=np.number).copy()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Features:", list(X.columns))
print("Classes:", list(label_encoder.classes_))


In [ ]:
# 14. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.20,
    stratify=y_encoded,
    random_state=RANDOM_STATE
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## Multiple ML Models

In [ ]:
# 15. Define models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_split=2,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3,
        random_state=RANDOM_STATE
    )
}

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE
    )
    print("XGBoost available — included.")
except ImportError:
    print("XGBoost not installed — continuing with sklearn models.")


In [ ]:
# 16. Train and evaluate all models
results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, pred, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)
display(results_df.round(4))


In [ ]:
# 17. Model comparison visualization
plot_df = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(13, 6))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.ylim(0, 1.05)
plt.title("ML Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# 18. Cross-validation comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)
    cv_rows.append({
        "Model": name,
        "CV_F1_Mean": scores.mean(),
        "CV_F1_Std": scores.std()
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV_F1_Mean", ascending=False)
display(cv_df.round(4))

plt.figure(figsize=(11, 6))
sns.barplot(data=cv_df, x="Model", y="CV_F1_Mean")
plt.errorbar(
    x=np.arange(len(cv_df)),
    y=cv_df["CV_F1_Mean"],
    yerr=cv_df["CV_F1_Std"],
    fmt="none", capsize=5
)
plt.ylim(0, 1.05)
plt.title("5-Fold Cross-Validation F1 Score")
plt.xlabel("Model")
plt.ylabel("Weighted F1")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# 19. Detailed classification report for the best model
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)

print("Best model:", best_name)
print()
print(classification_report(
    y_test, best_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))


In [ ]:
# 20. Confusion matrix
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
# 21. Feature importance for tree-based models
tree_model_name = next(
    (name for name in ["XGBoost", "Random Forest", "Gradient Boosting"] if name in fitted_models),
    None
)

if tree_model_name:
    tree_model = fitted_models[tree_model_name]

    # XGBoost/RandomForest/GradientBoosting expose feature_importances_
    importances = tree_model.feature_importances_
    fi = pd.DataFrame({
        "Feature": X.columns,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    display(fi.round(4))

    plt.figure(figsize=(10, 7))
    sns.barplot(data=fi, x="Importance", y="Feature")
    plt.title(f"Feature Importance — {tree_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("No tree-based model available.")


In [ ]:
# 22. ROC curves (one-vs-rest) for models that provide probabilities
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=np.arange(len(label_encoder.classes_)))

plt.figure(figsize=(10, 7))

for name, model in fitted_models.items():
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)

        # Plot micro-average ROC for compact comparison
        fpr, tpr, _ = roc_curve(y_test_bin.ravel(), proba.ravel())
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Micro-Average ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()


## Engineering Interpretation

### What the model is learning

For bearing diagnosis, vibration features can capture changes caused by defects:
- **RMS / standard deviation:** overall vibration energy
- **Peak / peak-to-peak:** impulsive vibration magnitude
- **Kurtosis:** sensitivity to impulsive events
- **Crest factor:** relationship between peak and RMS
- **RPM and load:** operating condition, which can influence vibration

For a real experimental project, you should also include frequency-domain features based on FFT and, ideally, bearing characteristic frequencies such as BPFO, BPFI, BSF, and FTF.

### Important research caution

Do not randomly split highly overlapping windows from the same original vibration recording into train and test sets. That can cause data leakage and unrealistically high accuracy. For real CWRU data, prefer a split by **recording/run or operating condition**, depending on the research question.

In [ ]:
# 23. Save results
results_df.to_csv("model_comparison_results.csv", index=False)
cv_df.to_csv("cross_validation_results.csv", index=False)

print("Saved:")
print("- model_comparison_results.csv")
print("- cross_validation_results.csv")


In [ ]:
# 24. Save the best model and label encoder
import joblib

joblib.dump(best_model, "best_bearing_fault_model.joblib")
joblib.dump(label_encoder, "bearing_fault_label_encoder.joblib")

print(f"Saved best model: {best_name}")


In [ ]:
# 25. Example prediction on one test observation
sample = X_test.iloc[[0]]
prediction_encoded = best_model.predict(sample)[0]
prediction_label = label_encoder.inverse_transform([prediction_encoded])[0]

print("Predicted bearing condition:", prediction_label)
display(sample)


## Suggested Final-Year Project Extension

To make this a stronger Mechanical Engineering project:

1. Replace the convenience CSV with the real CWRU vibration signals.
2. Segment raw acceleration into fixed windows.
3. Add FFT features and bearing characteristic-frequency amplitudes.
4. Compare time-domain ML with frequency-domain ML.
5. Add a real accelerometer + DAQ/ESP32 setup.
6. Build a Streamlit dashboard for real-time bearing health.
7. For an advanced version, add **Remaining Useful Life (RUL)** prediction using run-to-failure data.
